In [1]:
!pip install -q --upgrade \
    pdfplumber \
    tqdm

!apt-get install -y poppler-utils

# !apt-get update
# !apt-get install -y poppler-utils

!pip install opencv-python pdf2image numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 102.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/li

# Đại Việt Sử Ký Toàn Thư

In [8]:
import re
import json
import pdfplumber
from tqdm import tqdm
import cv2
import numpy as np
from pdf2image import convert_from_path

# PDF_PATH      = "test.pdf"
PDF_PATH      = "DVSK.pdf"
OUTPUT_JSON   = "DVSKTT_Chunks.json"
BOOK_NAME     = "Đại Việt Sử Ký Toàn Thư"

In [9]:
# @title

SEPARATORS = [
    "\n\n",
    "\n",
    ". ",
    "! ", "? ", "; ", ", ", " ", "",
]

PAGE_MARKER_FMT = "\n[[[PAGE:{page}]]]\n"

HEADER_FOOTER_PATTERNS = [
    re.compile(r'^\s*\d{1,4}\s+Đại Việt Sử Ký Toàn Thư.*$', re.MULTILINE | re.IGNORECASE),
    re.compile(r'^Đại Việt Sử Ký Toàn Thư.*\d{1,4}\s*$',    re.MULTILINE | re.IGNORECASE),
    re.compile(r'^\s*\d{1,4}\s*$', re.MULTILINE),
    re.compile(r'^\s*(Đại Việt Sử Ký Toàn Thư|Bản Kỷ|Ngoại Kỷ|Quyển\s+[IVXLC]+)\s*$',
               re.MULTILINE | re.IGNORECASE),
]

BOOK_PAGE_RE = re.compile(
    r'^\s*(\d{1,4})\s+Đại Việt Sử Ký Toàn Thư',
    re.MULTILINE | re.IGNORECASE
)

BOOK_HEADER_RE = re.compile(
    r'^\s*(?:\d{1,4}\s+)?(Đại Việt Sử Ký Toàn Thư[^\n\r]*)',
    re.MULTILINE | re.IGNORECASE
)



FOOTNOTE_LINE_RE = re.compile(
    r'^\s{0,9}(\d{1,2})(?:\.|\)|\]|\s{1,4}[A-ZÀ-Ỹ\[])'
)


# def replace_superscripts(text: str) -> str:
#     text = re.sub(r'([a-zA-ZÀ-ỹ\)])(\d+)(?=\s|[,.\!\?;:\)\]\n]|$)', r'\1 [\2]', text)
#     return text

def replace_superscripts(text: str) -> str:
    text = re.sub(r'([a-zA-ZÀ-ỹ\)\.])(\d+)(?=\s|[,.\!\?;:\)\]\n]|$)', r'\1 [\2]', text)
    return text

# header and footer
def remove_headers(text: str) -> str:
    for p in HEADER_FOOTER_PATTERNS:
        text = p.sub('', text)
    return text

def book_header(text: str) -> str:
    m = BOOK_HEADER_RE.search(text)

    if m:
        return(m.group(1).strip())
    return None


def extract_book_page_number(text: str):

    m = BOOK_PAGE_RE.search(text)

    if m:
        return int(m.group(1))

    return None

def normalize_whitespace(text: str) -> str:
    text = re.sub(r'[ \t]+\n', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'(?<![.!?;:])\n(?!\n)', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()



# ---  TÌM VỊ TRÍ ĐƯỜNG KẺ BẰNG OPENCV ---
def find_footnote_line_y(page_idx):
    """
    Chuyển trang PDF thành ảnh, dùng OpenCV quét tìm đường nằm ngang.
    Trả về tỷ lệ phần trăm vị trí Y trên trang (0.0 đến 1.0) để map với pdfplumber.
    """
    try:
        # Chuyển trang PDF cụ thể thành ảnh (DPI=150 là đủ tốt và nhanh)
        images = convert_from_path(PDF_PATH, first_page=page_idx, last_page=page_idx, dpi=150)
        if not images:
            return None

        img = np.array(images[0])
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        h_img, w_img = gray.shape

        # Nhận diện đường thẳng nằm ngang
        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)
        horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
        detect_horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)

        cnts = cv2.findContours(detect_horizontal, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = cnts[0] if len(cnts) == 2 else cnts[1]

        for c in cnts:
            x, y, w, h = cv2.boundingRect(c)
            # Lọc đường kẻ đủ dài (ở đây lớn hơn 15% độ rộng trang ảnh)
            w_ratio = w / w_img

            # Kiểm tra xem tỷ lệ đó có > 0.1 không
            if w_ratio >= 0.10 and (h_img / 2) <= y < (h_img * 0.99):
                # print(f"Tìm thấy đường kẻ phù hợp! Tỷ lệ rộng: {w_ratio:.2f}, Vị trí Y: {y} (Thuộc nửa dưới trang)")
                return y / h_img
    except Exception as e:
        print(f"Lỗi ở trang {page_idx}: {e}")
    return None


def extract_footnotes_by_layout(page, y_ratio) -> tuple[str, dict]:
    """
    Dựa vào tỷ lệ Y của đường kẻ, cắt không gian trang thành 2 nửa:
    Nửa trên -> Main Text, Nửa dưới -> Footnote Text
    """
    height = page.height
    width = page.width

    # Tính toán tọa độ thực tế trên đối tượng PDF (Bỏ qua 5 pixel nhiễu xung quanh đường kẻ)
    split_y = height * y_ratio

    # Cắt phần trên: từ y=0 đến y=split_y - 5
    main_bbox = (0, 0, width, split_y - 5)
    main_area = page.within_bbox(main_bbox)
    main_text = main_area.extract_text(x_tolerance=3, y_tolerance=3) if main_area else ""

    # Cắt phần dưới: từ y=split_y + 5 đến hết trang
    footer_bbox = (0, split_y + 5, width, height)
    footer_area = page.within_bbox(footer_bbox)
    footer_text = footer_area.extract_text(x_tolerance=3, y_tolerance=3) if footer_area else ""

    # Sử dụng lại logic phân tách từng footnote cũ dựa trên chuỗi chữ của footer_text
    footnote_dict = {}
    if footer_text.strip():
        footer_text = re.sub(r'\b\d{2}(?=\n)', r'\g<0> ', footer_text)
        # # Sau đó xóa dấu \n thừa ngay sau khoảng trắng đó để kéo dòng dưới lên
        footer_text = re.sub(r'(\b\d{2} )\n', r'\1', footer_text)
        lines = footer_text.split('\n')
        print(lines)
        # lines = footer_text
        current_key = None
        current_parts = []

        for line in lines:
            m = FOOTNOTE_LINE_RE.match(line)
            if m:
                if current_key is not None:
                    footnote_dict[current_key] = ' '.join(p.strip() for p in current_parts if p.strip())
                current_key = m.group(1)
                content_line = re.sub(r'^\s*\d{1,2}\s{1,4}', '', line).strip()
                current_parts = [content_line]
            elif current_key is not None:
                current_parts.append(line)

        if current_key is not None:
            footnote_dict[current_key] = ' '.join(p.strip() for p in current_parts if p.strip())

    return main_text, footnote_dict


def clean_page_v2(page, page_idx, raw_text: str) -> tuple[str, dict]:
    if not raw_text:
        return "", {}

    #  Thử tìm đường kẻ cắt trang bằng OpenCV
    y_ratio = find_footnote_line_y(page_idx)

    if y_ratio is not None:
        # Nếu tìm thấy đường kẻ: Cắt đôi trang dựa trên tọa độ địa lý hình ảnh
        text, fn_dict = extract_footnotes_by_layout(page, y_ratio)
    else:
          text = raw_text
          fn_dict = {}


    text = replace_superscripts(text)
    text = re.sub(r'\[\d{1,2}[ab]\]', '', text)
    # print(text)
    text = remove_headers(text)
    text = normalize_whitespace(text)

    return text, fn_dict


def extract_all_pages(pdf_path: str) -> list[dict]:
    # START_PAGE = 430
    # END_PAGE   = 450
    results = []
    with pdfplumber.open(pdf_path) as pdf:
        total = len(pdf.pages)
        print(f" Mở PDF: {total} trang")
        for i, page in enumerate(tqdm(pdf.pages, desc="📄 Đọc trang", unit="tr"), start=1):
            # if i < START_PAGE:
            #     continue
            # if i >= END_PAGE:
            #     break

            raw = page.extract_text(x_tolerance=3, y_tolerance=3)
            if not raw or len(raw.strip()) < 20:
                continue
        # lấy số trang ở header
            # print(raw)
            book_page = extract_book_page_number(raw)
            book_detail_name = book_header(raw)
            # print(f"đang đọc trang {book_page}")

            text, fn_dict = clean_page_v2(page, i, raw)

            if not text or len(text.strip()) < 20:
                continue
            results.append({"page_num": book_page, "book_header": book_detail_name, "text": text, "footnotes": fn_dict})
    return results


def build_full_text(pages: list[dict]) -> tuple[str, dict]:

    parts = []
    page_fn_map = {}
    page_headers_map = {}

    for page in pages:
        pn = page["page_num"]
        page_fn_map[pn] = page["footnotes"]
        # Đặt marker trước mỗi trang để track page boundary sau split
        parts.append(PAGE_MARKER_FMT.format(page=pn))
        parts.append(page["text"])

        page_headers_map[pn] = page.get("book_header")

    return "".join(parts), page_fn_map, page_headers_map


In [10]:

def main():
    print("  XỬ LÝ PDF - ĐẠI VIỆT SỬ KÝ TOÀN THƯ ")

# chuyển từ pdf -> json. Tải về 2 file.
# 1 file chứa các trang đơn lẻ.
# 1 file ghép toàn bộ trang lại với nhau.
    pages = extract_all_pages(PDF_PATH)

    with open("pages_DVSK_data.json", "w", encoding="utf-8") as f:
    # with open("test.json", "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

    print("Đã lưu pages_data.json")



    full_text, page_fn_map, page_headers_map = build_full_text(pages)

    full_text_data = {
        "book_name": BOOK_NAME,
        "full_text": full_text,
        "page_footnotes_map": page_fn_map,
        "page_headers_map": page_headers_map
    }

    with open("full_DVSK_data.json", "w", encoding="utf-8") as f:
    # with open("full_test.json", "w", encoding="utf-8") as f:
        json.dump(full_text_data, f, ensure_ascii=False, indent=2)

    print(" Đã lưu full_text_data.json")

    from google.colab import files

    files.download("pages_DVSK_data.json")
    files.download("full_DVSK_data.json")


if __name__ == "__main__":
    main()


  XỬ LÝ PDF - ĐẠI VIỆT SỬ KÝ TOÀN THƯ 
 Mở PDF: 739 trang


📄 Đọc trang:   0%|          | 1/739 [00:00<01:32,  8.00tr/s]

['Bản in Nội Các Quan Bản', 'Mộc bản khắc năm Chính Hòa thứ 18 (1697)']


📄 Đọc trang:   0%|          | 3/739 [00:00<03:28,  3.53tr/s]

['1 Hy thị: Tương truyền vua Nghiêu sai anh em họ Hy (Hy thị) và họ Hòa (Hòa thị) đi bốn phương để trông coi công việc thiên văn', 'lịch pháp. Hy Thúc là em Hy Trọng đến ở miền đất phương Nam (Kinh Thư, Nghiêu điển)', '2 Kinh Thư chép vua Nghiêu sai Hy Thúc đến ở Nam Giao (Hy Thúc trạch Nam Giao). Khổng An Quốc thời Tây Hán chú giải Kinh', 'Thư, chỉ cho Nam Giao là phương Nam. Mãi đến thời Đường, Tư Mã Trinh mới giải thích Nam Giao là Giao Chỉ ở phương Nam.', '3 Theo thiên Vũ Cống trong Kinh Thư, chín châu là Ký, Duyện, Thanh, Từ, Dương, Kinh, Dự, Lương Ung.', '4 Bách Việt: là từ mà người Hán dùng để gọi chung các tộc người khác Hán sống ở miền nam Trung Quốc thời xưa. Từ này lần đầu', 'tiên thấy chép trong Sử Ký (Ngô Khởi Truyện của Tư Mã Thiên.', '5 Việt Thường Thị: tên nước thời cổ ở phía Nam Trung Quốc có quan hệ với nhà Chu (hiến chim trĩ cho Thành Vương), lần đầu tiên', 'được ghi trong sách Thượng Thư Đại Truyện. Có nhiều giải thích khác nhau, có thuyết nói rằng Việt Thường Thị ở

...

['1 Quyển XXII chỉ có tờ 18 bị đóng nhầm vào vị trí tờ 18 quyển XXI.', '2 Bản Lê Hy không chép, nhưng ý này có trong Lịch triều hiến chương loại chí (Nhân vật chí) của Phan Huy Chú, đoạn nói về Lê Thì', 'HIến.', '3 Bản Lê Hy không chép sự kiện này.', '4 Tờ 18b kết thúc giữa câu với từ Bá.']
Đã lưu pages_data.json
 Đã lưu full_text_data.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>